# Capstone — Production Data Pipeline — Orientation Only 🔵
**Phase 2 | Project Prometheus**

---

## What This Is

The Phase 2 Capstone is a **production-grade data pipeline** that orchestrates all the skills from this phase into reusable infrastructure for Module 7 strategy work.

It was designed as an infrastructure project, not a portfolio project — its value is that Module 7 strategies can plug straight into it without rebuilding the data layer.

> **We're not building this as a standalone piece.** You already have a strong project list.
> This orientation tells you what it is, why it matters conceptually, and what to keep in mind when you reach Module 7.

---

## The Eight Modules (Awareness Level)

| Module | What it does | Key concept |
|--------|-------------|-------------|
| `MarketDataFetcher` | Pull multi-ticker data from yfinance | Error handling, retry logic, config-driven |
| `DataValidator` | Flag gaps, outliers, alignment issues | Catch bugs early — before they corrupt downstream |
| `CorporateActionHandler` | Validate splits/dividends | Unit 7 in production — zero-tolerance testing |
| `DataCleaner` | Missing data with documented decisions | Unit 8 in production — different logic per data type |
| `FeatureEngine` | Returns, vol, correlations, rankings | Unit 4+6 in production — extensible for Module 7 |
| `DataStore` | Persist to Parquet with metadata | Reproducibility — same input = same output |
| `QualityReport` | Automated data quality dashboard | Catches 95% of common errors |
| `Pipeline` | Orchestrates all of the above | CLI interface, logging, idempotent |

---

## Why Parquet (Not CSV)?

| Feature | CSV | Parquet |
|---------|-----|---------|
| Storage | Large | Compressed (10x smaller) |
| Read speed | Slow | Fast (columnar format) |
| Type preservation | Loses dtypes | Preserves all dtypes |
| Partial reads | Full file | Read specific columns only |
| Industry standard | ❌ | ✅ (used everywhere in quant) |

---

## Module 7 Compatibility Note

When you reach Module 7, your strategies will expect:
- **Multi-ticker data**: `pd.DataFrame` with `DatetimeIndex`, tickers as columns
- **Clean, adjusted prices**: `auto_adjust=True`, forward-filled gaps
- **Pre-computed features**: Returns (1d, 5d, 20d), vol (20d, 60d), cross-sectional ranks
- **Parquet format**: For fast loading with full type preservation

Even if you don't build the full pipeline, you should have these outputs ready.

---

## Minimum Viable Version (If You Want One)

If you want the Module 7 foundation without building 8 classes, a single notebook that does the following is sufficient:


In [ ]:
# Minimum viable data pipeline — Phase 2 → Module 7 bridge
import yfinance as yf
import pandas as pd
import numpy as np

# ── Config ──
TICKERS = ['SPY', 'QQQ', 'XLF', 'XLK', 'XLE', 'XLV', 'XLI', 'XLP', 'XLY', 'XLU', 'XLB']
START = '2019-01-01'
ROLL_SHORT = 20
ROLL_LONG = 60
TRADING_DAYS = 252

# ── Fetch ──
print("Fetching data...")
raw = yf.download(TICKERS, start=START, auto_adjust=True)['Close']
print(f"Raw shape: {raw.shape}")

# ── Validate ──
missing_pct = raw.isna().mean() * 100
bad_tickers = missing_pct[missing_pct > 5].index.tolist()
if bad_tickers:
    print(f"⚠️ High missingness (>5%): {bad_tickers}")

# ── Clean ──
prices = raw.ffill()  # Forward-fill price gaps (documented decision)

# ── Features ──
log_ret = np.log(prices / prices.shift(1))

features = pd.DataFrame()
for t in TICKERS:
    features[f'{t}_ret_1d']  = log_ret[t]
    features[f'{t}_ret_5d']  = log_ret[t].rolling(5).sum()
    features[f'{t}_ret_20d'] = log_ret[t].rolling(20).sum()
    features[f'{t}_vol_20d'] = log_ret[t].rolling(ROLL_SHORT).std() * np.sqrt(TRADING_DAYS)
    features[f'{t}_vol_60d'] = log_ret[t].rolling(ROLL_LONG).std() * np.sqrt(TRADING_DAYS)

features = features.dropna()
print(f"\nFeature matrix shape: {features.shape}")
print(features.tail(3))


In [ ]:
# ── Persist to Parquet ──
import os

output_dir = 'data'
os.makedirs(output_dir, exist_ok=True)

prices.to_parquet(f'{output_dir}/prices_clean.parquet')
features.to_parquet(f'{output_dir}/features.parquet')

print("Saved:")
print(f"  {output_dir}/prices_clean.parquet")
print(f"  {output_dir}/features.parquet")

# ── Reload and verify ──
prices_check = pd.read_parquet(f'{output_dir}/prices_clean.parquet')
print(f"\nReloaded prices shape: {prices_check.shape}")
print("✅ Parquet round-trip verified")


---

## Interview Framing

If asked about your data infrastructure in an interview:

> *"I built a modular data pipeline covering ingestion, validation, corporate action handling, and feature engineering. The pipeline outputs Parquet files with a full audit trail — source, version, cleaning decisions. I designed it so strategy modules can plug straight in without touching the data layer."*

Even if you only built the minimum viable version above — that answer is truthful and demonstrates production thinking.

---

## Phase 2 Exit Criteria ✅

You're ready for Phase 3 when:
- [ ] You can explain rolling volatility and variance scaling from memory
- [ ] You know when to use `.ffill()` vs leave as NaN (by data type)
- [ ] You can spot survivorship bias in someone else's backtest description
- [ ] You understand what `.transform()` does vs `.agg()`
- [ ] You know what `auto_adjust=True` does and why it matters
- [ ] You can defend every line of your data cleaning code in an interview
